# Autoencoders e Representações Latentes

Neste notebook, exploraremos como redes neurais artificiais podem aprender representações compactas e semanticamente ricas de dados em **espaços latentes** de menor dimensionalidade de forma totalmente não supervisionada. 

A ideia central do aprendizado de representações é projetar entradas de alta dimensionalidade (como imagens contendo centenas ou milhares de pixels) em vetores compactos (chamados de variáveis latentes ou *embeddings*). Ao forçar os dados a passar por esse "gargalo", o modelo é obrigado a reter apenas as características fundamentais e estruturais que melhor descrevem a variabilidade dos dados originais.

### O Espaço Latente

Formalmente, definimos o processo através de duas funções principais:
1. **Codificador ($f_\theta$):** Mapeia um dado de entrada $\mathbf{x} \in \mathbb{R}^D$ para um vetor no espaço latente $\mathbf{z} \in \mathbb{R}^m$, onde tipicamente $m \ll D$.
   $$\mathbf{z} = f_\theta(\mathbf{x})$$
2. **Decodificador ($g_\phi$):** Recebe o vetor latente $\mathbf{z}$ e tenta mapeá-lo de volta para o espaço original, produzindo uma reconstrução $\mathbf{\hat{x}} \in \mathbb{R}^D$.
   $$\mathbf{\hat{x}} = g_\phi(\mathbf{z})$$

Onde $\theta$ e $\phi$ representam os parâmetros (pesos e vieses) das respectivas redes neurais.

Neste notebook, implementaremos uma arquitetura clássica de **Autoencoder** utilizando PyTorch e o dataset MNIST. Analisaremos a qualidade de suas reconstruções, visualizaremos o espaço latente aprendido através do algoritmo **t-SNE** e realizaremos **interpolações lineares** para explorar a continuidade semântica do espaço latente. 

Por fim, estudaremos e implementaremos **Denoising Autoencoders (DAEs)**, uma variação projetada para remover ruído e aprender representações ainda mais robustas e estáveis sobre a variedade dos dados.

In [ ]:
import torch
from torch import nn
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo utilizado: {device}")

### Preparação dos Dados (MNIST)

Para este estudo didático, utilizaremos o dataset **MNIST**, composto por imagens em tons de cinza de $28 \times 28$ pixels contendo dígitos manuscritos de 0 a 9. 

Para assegurar que o treinamento seja extremamente rápido e possa ser executado sem problemas mesmo em ambientes sem aceleração por hardware (GPU), utilizaremos a classe `Subset` do PyTorch para extrair uma amostra de **10.000 imagens para treino** e **1.000 imagens para validação**. Isso reduz drasticamente o tempo de treinamento sem sacrificar o aprendizado dos conceitos essenciais do modelo.


In [ ]:
transform = transforms.ToTensor()

train_dataset = datasets.MNIST("./data", train=True, download=True, transform=transform)
val_dataset = datasets.MNIST("./data", train=False, download=True, transform=transform)

train_subset = Subset(train_dataset, range(10000))
val_subset = Subset(val_dataset, range(1000))

train_loader = DataLoader(train_subset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=64)

print(len(train_subset), "amostras de treino")
print(len(val_subset), "amostras de validação")

## Arquitetura do Autoencoder

O Autoencoder utilizado neste notebook é formado por duas partes principais: um codificador (*encoder*) e um decodificador (*decoder*). O codificador transforma a imagem de entrada em uma representação latente compacta, enquanto o decodificador utiliza essa representação para reconstruir uma aproximação da imagem original.

Neste exemplo, as imagens do MNIST são inicialmente representadas como vetores em $\mathbb{R}^{784}$, correspondentes aos pixels de uma imagem $28 \times 28$ achatada. O codificador reduz progressivamente essa representação até um vetor latente de dimensão $m = 16$. Essa camada de menor dimensão funciona como um gargalo, forçando o modelo a preservar apenas as informações mais relevantes para a reconstrução.

O decodificador realiza o processo inverso: parte do vetor latente em $\mathbb{R}^{16}$ e produz uma saída novamente em $\mathbb{R}^{784}$, que pode ser reorganizada no formato original da imagem. As camadas intermediárias utilizam ativações ReLU, permitindo que o modelo aprenda transformações não lineares entre os espaços de entrada, latente e reconstruído.

A última camada utiliza uma ativação Sigmoid, pois as imagens foram normalizadas para o intervalo $[0, 1]$. Assim, os valores produzidos pelo modelo também ficam compatíveis com a escala das intensidades dos pixels da imagem original.

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, encoding_dim=16):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(28 * 28, 64),
            nn.ReLU(),
            nn.Linear(64, encoding_dim)
        )

        self.decoder = nn.Sequential(
            nn.Linear(encoding_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 28 * 28),
            nn.Sigmoid()
        )

    def forward(self, x):
        z = self.encoder(x)
        x_recon = self.decoder(z)
        return x_recon

### Instanciando o Modelo

Instanciaremos o modelo especificando um espaço latente com dimensão de 16 representações (`latent_dim = 16`). Em seguida, transferiremos a rede neural para o dispositivo configurado (`device`).


In [ ]:
latent_dim = 16
ae_model = Autoencoder(encoding_dim=latent_dim).to(device)
print(ae_model)

### Função de Perda (Loss) e Otimizador

Como o Autoencoder tradicional aprende de forma não supervisionada mapeando os dados para si mesmos, a **função de perda** mede o erro de reconstrução entre a imagem original $\mathbf{x}$ e a reconstrução produzida pelo decodificador $\mathbf{\hat{x}}$.

Utilizaremos o **Erro Quadrático Médio (MSE Loss)**, que calcula a média das diferenças quadradas de pixel por pixel:

$$\mathcal{L}_{MSE}(\mathbf{x}, \mathbf{\hat{x}}) = \frac{1}{D} \sum_{j=1}^{D} (x_j - \hat{x}_j)^2$$

onde $D = 784$. Utilizaremos o algoritmo **Adam** para otimizar os parâmetros de ambas as sub-redes em conjunto.


In [ ]:
ae_criterion = nn.MSELoss()
ae_optimizer = torch.optim.Adam(ae_model.parameters(), lr=1e-3, weight_decay=1e-8)

### Loop de Treinamento

No processo de treinamento, iteramos por 20 épocas. A cada iteração:
1. Achatamos as imagens de entrada de formato $28 \times 28$ para vetores 1D de 784 elementos.
2. Executamos a passagem direta (forward propagation) para obter a reconstrução.
3. Calculamos a perda de reconstrução (MSE Loss) contra as imagens originais limpas.
4. Realizamos a retropropagação (backpropagation) e atualizamos os pesos com o otimizador Adam.


In [ ]:
epochs = 20
ae_model.train()

train_losses = []

for epoch in range(epochs):
    epoch_loss = 0.0
    for X, _ in train_loader:
        # Achatamento e envio ao dispositivo
        X = X.view(X.size(0), -1).to(device)
        
        # Forward pass
        recon = ae_model(X)
        loss = ae_criterion(recon, X)

        # Backward pass e otimização
        ae_optimizer.zero_grad()
        loss.backward()
        ae_optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    
    print(f"[{epoch+1}/{epochs}] Loss: {avg_loss:.4f}")

# Plotagem da perda histórica com estilo didático
plt.figure(figsize=(8, 5))
plt.plot(range(1, epochs + 1), train_losses, marker="o", color="royalblue", linewidth=2, label="Loss (MSE)")
plt.title("Histórico de Perda do Autoencoder durante o Treinamento")
plt.xlabel("Época")
plt.ylabel("Erro Quadrático Médio (MSE)")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.show()

## Análise de Reconstrução

Uma das formas mais diretas de avaliar qualitativamente o desempenho do Autoencoder é passar um batch de imagens de teste e plotá-las lado a lado com suas respectivas reconstruções. Isso nos permite ver quais características (como formas arredondadas, inclinações e traços estruturais) foram preservadas com precisão e quais detalhes sutis de alta frequência foram ligeiramente atenuados devido ao gargalo latente de 16 dimensões.


In [ ]:
# Avaliação qualitativa do modelo
ae_model.eval()
n = 10

with torch.no_grad():
    # Obtém um batch de dados de validação
    data_iter = iter(val_loader)
    images, _ = next(data_iter)
    images_flat = images.view(images.size(0), -1).to(device)
    
    # Gera reconstruções
    reconstructed_flat = ae_model(images_flat)
    
    # Conversão para numpy para plotagem
    original_images = images.cpu().numpy()
    reconstructed_images = reconstructed_flat.view(-1, 1, 28, 28).cpu().numpy()

In [ ]:
plt.figure(figsize=(18, 5))

for i in range(n):
    # Imagem original
    ax = plt.subplot(2, n, i + 1)
    plt.imshow(original_images[i].squeeze(), cmap='gray', vmin=0, vmax=1)
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
    if i == n // 2:
        ax.set_title('Imagens Originais (Input)', fontsize=12, pad=10)

    # Imagem reconstruída
    ax = plt.subplot(2, n, i + 1 + n)
    plt.imshow(reconstructed_images[i].squeeze(), cmap='gray', vmin=0, vmax=1)
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
    if i == n // 2:
        ax.set_title('Imagens Reconstruídas (Output)', fontsize=12, pad=10)

plt.tight_layout()
plt.show()

### Visualização do Espaço Latente com t-SNE

Como o espaço latente do nosso Autoencoder possui 16 dimensões, não é possível plotá-lo diretamente de forma bidimensional. Para isso, utilizamos o algoritmo **t-SNE** (t-Distributed Stochastic Neighbor Embedding), uma técnica de redução de dimensionalidade não linear que preserva relações de proximidade no espaço de origem.

Interessantemente, mesmo sem fornecer qualquer rótulo de classe (dígito) ao Autoencoder durante o treinamento (aprendizado puramente não supervisionado), observaremos que dígitos semelhantes se agrupam espontaneamente em clusters contínuos no espaço latente de 16 dimensões devido às suas similaridades geométricas!


In [ ]:
ae_model.eval()
ae_all_latents = []
ae_all_labels = []

with torch.no_grad():
    for X, y in val_loader:
        X = X.view(X.size(0), -1).to(device)
        latent = ae_model.encoder(X)
        ae_all_latents.append(latent.cpu().numpy())
        ae_all_labels.append(y.cpu().numpy())

ae_latent_space_test = np.concatenate(ae_all_latents, axis=0)
ae_labels_test = np.concatenate(ae_all_labels, axis=0)

# Reduzindo para 2 dimensões com t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
tsne_results = tsne.fit_transform(ae_latent_space_test)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(tsne_results[:, 0], tsne_results[:, 1], c=ae_labels_test, cmap='tab10', s=15, alpha=0.8)
plt.colorbar(scatter, label='Dígito Real')
plt.title('Projeção t-SNE do Espaço Latente', fontsize=14)
plt.xlabel('Componente t-SNE 1')
plt.ylabel('Componente t-SNE 2')
plt.grid(True, linestyle="--", alpha=0.3)
plt.show()

## Interpolando no Espaço Latente

Uma evidência crucial de que o Autoencoder aprendeu uma representação semanticamente contínua dos dados, em vez de apenas memorizar amostras isoladas, é a capacidade de realizar **interpolação linear** entre dois vetores latentes $\mathbf{z}_1$ e $\mathbf{z}_2$.

Dada a fórmula de interpolação paramétrica:

$$\mathbf{z}(t) = (1 - t) \mathbf{z}_1 + t \mathbf{z}_2, \quad t \in [0, 1]$$

onde $t$ varia linearmente de 0 a 1. Passamos cada vetor interpolado $\mathbf{z}(t)$ pelo decodificador $g_\phi$ e visualizamos a transição suave de características de um dígito para o outro (por exemplo, a transformação morfológica de um número em outro).


In [ ]:
ae_model.eval()

imgs, labels = next(iter(val_loader))

idx1 = 0
idx2 = next(i for i in range(len(labels)) if labels[i] != labels[idx1])

x1 = imgs[idx1].view(1, -1).to(device)
x2 = imgs[idx2].view(1, -1).to(device)

label1 = labels[idx1].item()
label2 = labels[idx2].item()

with torch.no_grad():
    z1 = ae_model.encoder(x1)
    z2 = ae_model.encoder(x2)

steps = 8
alphas = np.linspace(0, 1, steps)
interpolations = []

with torch.no_grad():
    for alpha in alphas:
        zt = (1 - alpha) * z1 + alpha * z2
        xt = ae_model.decoder(zt)
        xt = xt.view(28, 28).cpu().numpy()
        interpolations.append(xt)

fig, axes = plt.subplots(1, steps, figsize=(18, 3))

for ax, img, alpha in zip(axes, interpolations, alphas):
    ax.imshow(img, cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"{alpha:.2f}", fontsize=9)
    ax.axis("off")

fig.suptitle(
    f"Interpolação no Espaço Latente: dígito {label1} → dígito {label2}",
    fontsize=14,
    y=1.05
)

plt.tight_layout()
plt.show()

## Denoising Autoencoders (DAE)

Um **Denoising Autoencoder (DAE)** é uma variação do Autoencoder tradicional. Em vez de simplesmente mapear os dados idênticos da entrada para a saída, o DAE é projetado para **reconstruir um dado limpo a partir de uma versão intencionalmente corrompida por ruído**.

### Formulação

O processo funciona da seguinte forma:
1. Tomamos uma imagem original limpa $\mathbf{x} \in \mathbb{R}^D$.
2. Injetamos um processo estocástico de ruído para corrompê-la: $\mathbf{\tilde{x}} \sim q(\mathbf{\tilde{x}}|\mathbf{x})$. Por exemplo, adicionamos ruído gaussiano aditivo: $\mathbf{\tilde{x}} = \mathbf{x} + \boldsymbol{\epsilon}$, onde $\boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0}, \sigma^2 \mathbf{I})$.
3. O codificador mapeia a imagem ruidosa para o espaço latente: $\mathbf{z} = f_\theta(\mathbf{\tilde{x}})$.
4. O decodificador reconstrói a imagem limpa original a partir de $\mathbf{z}$: $\mathbf{\hat{x}} = g_\phi(\mathbf{z})$.

O modelo é treinado minimizando o erro de reconstrução **contra a imagem limpa original**:

$$\mathcal{L}_{DAE}(\mathbf{x}, \mathbf{\hat{x}}) = \mathcal{L}_{DAE}(\mathbf{x}, g_\phi(f_\theta(\mathbf{\tilde{x}}))) = \frac{1}{D} \sum_{j=1}^{D} (x_j - g_\phi(f_\theta(\mathbf{\tilde{x}}))_j)^2$$

Para remover o ruído e reconstruir a imagem original perfeitamente, o DAE não pode simplesmente memorizar a entrada por identidade. Em vez disso, o modelo deve aprender a **estrutura geométrica essencial e a topologia da variedade onde residem os dados reais (data manifold)**. 

O ruído afasta o dado de sua variedade original; o processo de "denoising" atua essencialmente como um operador de projeção, empurrando pontos ruidosos de volta para a região de alta probabilidade dos dados reais.

In [ ]:
# Função para injetar ruído Gaussiano limitando os pixels no intervalo aceitável [0, 1]
def add_gaussian_noise(x, noise_factor=0.4):
    noisy = x + noise_factor * torch.randn_like(x)
    return torch.clamp(noisy, 0.0, 1.0)

In [ ]:
# Instanciação de um novo modelo para Denoising Autoencoder
dae_model = Autoencoder(encoding_dim=latent_dim).to(device)
dae_criterion = nn.MSELoss()
dae_optimizer = torch.optim.Adam(dae_model.parameters(), lr=1e-3, weight_decay=1e-8)

dae_epochs = 20
dae_model.train()
dae_losses = []

print("Iniciando Treinamento do Denoising Autoencoder (DAE)...")
for epoch in range(dae_epochs):
    epoch_loss = 0.0
    for X, _ in train_loader:
        # Achatamento da imagem original
        X_flat = X.view(X.size(0), -1).to(device)
        
        # Corrompendo a entrada com ruído
        X_noisy = add_gaussian_noise(X_flat, noise_factor=0.4)
        
        # Forward pass: entrada é ruidosa, mas o alvo (target) é a original limpa!
        recon = dae_model(X_noisy)
        loss = dae_criterion(recon, X_flat)

        # Backward pass e otimização
        dae_optimizer.zero_grad()
        loss.backward()
        dae_optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader)
    dae_losses.append(avg_loss)
    print(f"[{epoch+1}/{dae_epochs}] Loss DAE: {avg_loss:.4f}")

### Avaliação da Remoção de Ruído (Denoising)

Podemos agora avaliar o DAE qualitativamente. Vamos alimentar o DAE com imagens ruidosas inéditas do conjunto de validação e exibir uma comparação visual composta por 3 linhas:
1. **Imagens Originais Limpas** (referência).
2. **Imagens Ruidosas** (entrada real alimentada ao DAE).
3. **Imagens Denoised** (saída produzida pelo DAE).


In [ ]:
dae_model.eval()
n = 10

with torch.no_grad():
    data_iter = iter(val_loader)
    images, _ = next(data_iter)
    images_flat = images.view(images.size(0), -1).to(device)
    
    # Corrompendo com ruído
    images_noisy = add_gaussian_noise(images_flat, noise_factor=0.4)
    
    # Removendo ruído via modelo
    denoised_flat = dae_model(images_noisy)
    
    # Conversões para numpy para exibição
    original = images.cpu().numpy()
    noisy = images_noisy.view(-1, 1, 28, 28).cpu().numpy()
    denoised = denoised_flat.view(-1, 1, 28, 28).cpu().numpy()

In [ ]:
# Plotagem comparativa de 3 níveis
plt.figure(figsize=(18, 7.5))

for i in range(n):
    # 1. Originais Limpas
    ax = plt.subplot(3, n, i + 1)
    plt.imshow(original[i].squeeze(), cmap='gray', vmin=0, vmax=1)
    ax.axis('off')
    if i == n // 2:
        ax.set_title('Imagens Originais Limpas', fontsize=12, pad=8)

    # 2. Entradas Ruidosas
    ax = plt.subplot(3, n, i + 1 + n)
    plt.imshow(noisy[i].squeeze(), cmap='gray', vmin=0, vmax=1)
    ax.axis('off')
    if i == n // 2:
        ax.set_title('Entradas Ruidosas (Alimentadas ao DAE)', fontsize=12, pad=8)

    # 3. Saídas Denoised
    ax = plt.subplot(3, n, i + 1 + 2 * n)
    plt.imshow(denoised[i].squeeze(), cmap='gray', vmin=0, vmax=1)
    ax.axis('off')
    if i == n // 2:
        ax.set_title('Saídas Denoised (Reconstruídas pelo DAE)', fontsize=12, pad=8)

plt.tight_layout()
plt.show()

## Exercícios

### Exercício 1: Compressão Extrema no Espaço Latente 2D

Treine uma versão do autoencoder usando `encoding_dim = 2` em vez de `encoding_dim = 16`. Depois, compare visualmente as reconstruções obtidas e analise como a redução extrema da dimensão latente afeta a qualidade das imagens reconstruídas.

Como o espaço latente agora possui apenas duas dimensões, visualize diretamente os vetores latentes do conjunto de validação em um gráfico de dispersão, usando as coordenadas $(z_1, z_2)$ e colorindo os pontos de acordo com a classe do dígito. Compare essa organização com a projeção t-SNE obtida anteriormente a partir do espaço latente de 16 dimensões.